# RF-DETR Single-Class Detection Pipeline: location_tag
### Model: RF-DETR Base (`resolution=560`, `optimizer=AdamW`, `lr=1e-4`, `grad_accumulation=8`)
This pipeline fine-tunes an RF-DETR Base model on `location_tag` using:
- **Exact User Configuration & Pipeline Architecture**: `load_coco_info`, `visualize_samples`, `_fixed_reinitialize` detection head patch, `COCODetectionDataset`, `build_criterion_and_postprocessors`, `BestValLossCheckpointSD`, AMP (`GradScaler`), and `MeanAveragePrecision`.
- **Sample vs Full Mode**: Toggle `SAMPLE_SIZE = 1000` for fast pipeline testing vs `SAMPLE_SIZE = None` for complete dataset training.
- **Effective Batch Size**: `BATCH_SIZE = 4` with `GRAD_ACCUMULATION = 8` (effective batch size = 32).
- **VRAM Monitor**: Real-time tracking of GPU allocated, reserved, peak, and free memory.

In [ ]:
# STEP 0 — Install Dependencies (RF-DETR 1.4.0, Torchmetrics & CUDA Stack)
print("=" * 70)
print("📦 [STEP 0] Verifying and installing dependencies...")
print("=" * 70)

!pip install -q --no-cache-dir "torch==2.5.1" "torchvision==0.20.1" --index-url https://download.pytorch.org/whl/cu121
!pip install -q --no-cache-dir "rfdetr==1.4.0"
!pip install -q --no-cache-dir "supervision>=0.22.0"
!pip install -q --no-cache-dir torchmetrics
!pip install -q --no-cache-dir pycocotools pandas numpy opencv-python Pillow matplotlib python-dotenv azure-storage-blob requests tqdm

print("✅ Dependencies verified successfully.")


In [ ]:
# CELL 1 — Imports, VRAM Monitor & Logger Initialization
import os
import sys
import json
import time
import math
import copy
import logging
import random
import shutil
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed

import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd
from PIL import Image
import requests
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from dotenv import load_dotenv
from tqdm.notebook import tqdm
import supervision as sv

try:
    from torchmetrics.detection.mean_ap import MeanAveragePrecision
    torchmetrics_status = "Available (MeanAveragePrecision)"
except ImportError:
    torchmetrics_status = "Not installed yet - run Step 0"

try:
    from rfdetr import RFDETRBase, RFDETRLarge
    from rfdetr import main as rfdetr_main
    from rfdetr.models.lwdetr import build_criterion_and_postprocessors
    rfdetr_status = "Available (RFDETRBase, RFDETRLarge)"
except ImportError:
    rfdetr_status = "Not installed yet - run Step 0"

logger = logging.getLogger("rfdetr_single_class")
logger.setLevel(logging.INFO)
logger.handlers.clear()

log_format = "%(asctime)s | %(levelname)-8s | %(message)s"
console_handler = logging.StreamHandler(sys.stdout)
console_handler.setFormatter(logging.Formatter(log_format))
logger.addHandler(console_handler)

def print_vram_usage(tag="Current Status"):
    """Displays real-time VRAM allocation, reservation, peak usage, and free memory."""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / (1024 ** 3)
        reserved = torch.cuda.memory_reserved() / (1024 ** 3)
        max_allocated = torch.cuda.max_memory_allocated() / (1024 ** 3)
        total = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        free = total - reserved
        print(f"📊 [VRAM Monitor — {tag}]")
        print(f"   • GPU Device:      {torch.cuda.get_device_name(0)}")
        print(f"   • Total VRAM:      {total:.2f} GB")
        print(f"   • Allocated VRAM:  {allocated:.2f} GB ({allocated/total*100:.1f}%)")
        print(f"   • Reserved Cache:  {reserved:.2f} GB ({reserved/total*100:.1f}%)")
        print(f"   • Peak Allocated:  {max_allocated:.2f} GB ({max_allocated/total*100:.1f}%)")
        print(f"   • Free VRAM:       {free:.2f} GB ({free/total*100:.1f}%)")
    else:
        print(f"📊 [VRAM Monitor — {tag}] Running on CPU (No CUDA GPU detected).")

print("=" * 70)
print("🚀 [CELL 1] Core Libraries & Logger Initialized")
print("=" * 70)
print(f"   • Python Version:      {sys.version.split()[0]}")
print(f"   • PyTorch Version:     {torch.__version__} (CUDA Available: {torch.cuda.is_available()})")
print(f"   • Supervision Version: {sv.__version__}")
print(f"   • Torchmetrics:        {torchmetrics_status}")
print(f"   • RF-DETR Module:      {rfdetr_status}")
print_vram_usage("Initial Environment Check")


In [ ]:
# CELL 2 — Configuration & Mode Setup
# ==============================================================================
# SAMPLE vs FULL DATA MODE:
# Set SAMPLE_SIZE = 1000 to sample 1,000 images for fast testing and flow verification.
# Set SAMPLE_SIZE = None to run on the complete full dataset.
# ==============================================================================
SAMPLE_SIZE = 1000  # Set to None for full dataset training

PIPELINE_NAME = "location_tag_single_class_train_rfdetr"
MODE_TAG = f"sample_{SAMPLE_SIZE}" if SAMPLE_SIZE else "full_data"

# Repository & Input directories
REPO_ROOT = Path(".")
PIPELINE_DIR = REPO_ROOT / PIPELINE_NAME
INPUT_JSON_DIR = REPO_ROOT / "coco_files"

# Dataset paths
DATASET_DIR = os.path.join(PIPELINE_DIR, f"dataset_{MODE_TAG}")
TRAIN_DIR = os.path.join(DATASET_DIR, "train")
VAL_DIR = os.path.join(DATASET_DIR, "val")
TEST_DIR = os.path.join(DATASET_DIR, "test")

TRAIN_ANN = os.path.join(TRAIN_DIR, "_annotations.coco.json")
VAL_ANN = os.path.join(VAL_DIR, "_annotations.coco.json")
TEST_ANN = os.path.join(TEST_DIR, "_annotations.coco.json")

# Model Parameters
MODEL_SIZE = "base"     # "base" (~29M params) or "large" (~128M params)
PRETRAINED = True
RESOLUTION = 560        # input resolution; must be divisible by 56

# Training Parameters
EPOCHS = 100
BATCH_SIZE = 4          # reduce to 2 if you hit OOM
GRAD_ACCUMULATION = 8   # effective batch = BATCH_SIZE * GRAD_ACCUMULATION = 32
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2

# Output & Checkpoint Parameters
OUTPUT_DIR = os.path.join(PIPELINE_DIR, f"runs/rf_detr_{MODE_TAG}")
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
LOG_DIR = os.path.join(OUTPUT_DIR, "logs")
INFERENCE_OUTPUT_DIR = os.path.join(PIPELINE_DIR, f"inference_{MODE_TAG}")
FINAL_MODEL_DIR = os.path.join(PIPELINE_DIR, "model")

PRETRAINED_WEIGHTS_CANDIDATES = [
    os.path.expanduser("~/rf-detr-base-coco.pth"),
    "/home/jupyter/rf-detr-base-coco.pth",
    "rf-detr-base-coco.pth"
]
PRETRAINED_WEIGHTS = next((p for p in PRETRAINED_WEIGHTS_CANDIDATES if os.path.exists(p)), "rf-detr-base-coco.pth")
AZURE_CONNECTION_STRING_ENV = "AZURE_STORAGE_CONNECTION_STRING"

IMAGE_FIELD = "image_id"
CATEGORY_FIELD = "category_id"
BBOX_FIELD = "bbox"
TARGET_CLASS = "location_tag"
NUM_CLASSES = 1

TRAIN_RATIO = 0.80
VALID_RATIO = 0.10
TEST_RATIO = 0.10
RANDOM_SEED = 42

DOWNLOAD_WORKERS = 16
CONFIDENCE = 0.50
NMS_THRESHOLD = 0.50

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("high")

print("=" * 70)
print("Configuration summary")
print("=" * 70)
print(f" Dataset dir       : {DATASET_DIR}")
print(f" Model size        : RF-DETR-{MODEL_SIZE.capitalize()}")
print(f" Resolution        : {RESOLUTION}x{RESOLUTION}")
print(f" Epochs            : {EPOCHS}")
print(f" Batch size        : {BATCH_SIZE} (effective: {BATCH_SIZE * GRAD_ACCUMULATION})")
print(f" Learning rate     : {LEARNING_RATE}")
print(f" Weight decay      : {WEIGHT_DECAY}")
print(f" Output dir        : {OUTPUT_DIR}")
print(f" Checkpoint dir    : {CHECKPOINT_DIR}")
print(f" Log dir           : {LOG_DIR}")
print("=" * 70)


In [ ]:
# CELL 3 — Dynamic Folder Creation & File Logging
print("=" * 70)
print("📁 [CELL 3] Creating pipeline directories dynamically...")
print("=" * 70)

for p in [DATASET_DIR, OUTPUT_DIR, CHECKPOINT_DIR, LOG_DIR, INFERENCE_OUTPUT_DIR, FINAL_MODEL_DIR]:
    os.makedirs(p, exist_ok=True)
    print(f"   📁 Ready: {p}")

log_file_path = os.path.join(LOG_DIR, "pipeline.log")
file_handler = logging.FileHandler(log_file_path, mode="a", encoding="utf-8")
file_handler.setFormatter(logging.Formatter(log_format))
logger.addHandler(file_handler)

load_dotenv()
connection_string = os.getenv(AZURE_CONNECTION_STRING_ENV)
try:
    from azure.storage.blob import BlobServiceClient
    blob_service_client = BlobServiceClient.from_connection_string(connection_string) if connection_string else None
    if blob_service_client:
        print("   ☁️  Azure BlobServiceClient initialized.")
    else:
        print("   ⚠️  Azure connection string not set. Will use local image files if available.")
except Exception as e:
    blob_service_client = None
    print(f"   ℹ️  Azure SDK note: {e}")

print(f"✅ Dynamic directories prepared. File logger writing to {log_file_path}")
logger.info(f"Dynamic directories created for {PIPELINE_NAME}.")


In [ ]:
# CELL 4 — Read all annotation JSON files from coco_files/
print("=" * 70)
print(f"📄 [CELL 4] Searching for annotation JSON files in: {INPUT_JSON_DIR}")
print("=" * 70)

if not INPUT_JSON_DIR.exists():
    INPUT_JSON_DIR.mkdir(parents=True, exist_ok=True)

json_files = sorted(list(INPUT_JSON_DIR.glob("*.json")))
print(f"   • Found {len(json_files)} JSON file(s) in {INPUT_JSON_DIR}")

raw_records = []
for jf in json_files:
    print(f"   • Reading: {jf.name} ({jf.stat().st_size / 1024:.1f} KB)")
    with open(jf, "r", encoding="utf-8") as f:
        content = f.read().strip()
        if not content:
            continue
        try:
            parsed = json.loads(content)
            if isinstance(parsed, list):
                raw_records.extend(parsed)
            elif isinstance(parsed, dict):
                if "annotations" in parsed and isinstance(parsed["annotations"], list):
                    raw_records.extend(parsed["annotations"])
                else:
                    raw_records.append(parsed)
        except json.JSONDecodeError:
            f.seek(0)
            for line_idx, line in enumerate(f):
                line = line.strip()
                if line:
                    try:
                        parsed_line = json.loads(line)
                        if isinstance(parsed_line, list):
                            raw_records.extend(parsed_line)
                        else:
                            raw_records.append(parsed_line)
                    except Exception as err:
                        logger.warning(f"Error decoding line {line_idx+1} in {jf.name}: {err}")

print(f"✅ Total raw annotation records loaded: {len(raw_records)}")
logger.info(f"Loaded {len(raw_records)} raw annotation records from {len(json_files)} file(s).")


In [ ]:
# CELL 5 — Parse Annotations & Record Original Categories
print("=" * 70)
print("🔍 [CELL 5] Parsing annotations and mapping to single-class ('location_tag')...")
print("=" * 70)

image_annotations: Dict[str, List[Dict[str, Any]]] = {}
original_category_set = set()

for item in raw_records:
    if not isinstance(item, dict):
        continue
    img_id = item.get(IMAGE_FIELD)
    if not img_id:
        continue
    
    raw_cat = item.get(CATEGORY_FIELD, ["location_tag"])
    if isinstance(raw_cat, list):
        cat_name = str(raw_cat[0]) if len(raw_cat) > 0 else "location_tag"
    else:
        cat_name = str(raw_cat)
    original_category_set.add(cat_name)

    raw_bbox = item.get(BBOX_FIELD, [])
    if not (isinstance(raw_bbox, list) and len(raw_bbox) == 4):
        continue
    
    try:
        x, y, w, h = [float(v) for v in raw_bbox]
        if w <= 0 or h <= 0:
            continue
    except (ValueError, TypeError):
        continue

    # Single-Class: Remap to class ID 0 ('location_tag')
    ann_dict = {
        "bbox": [x, y, w, h],
        "category_id": 0,
        "category_name": TARGET_CLASS,
        "original_category": cat_name,
        "area": float(item.get("area", w * h)),
        "segmentation": item.get("segmentation", [])
    }
    
    if img_id not in image_annotations:
        image_annotations[img_id] = []
    image_annotations[img_id].append(ann_dict)

ORIGINAL_CATEGORIES = sorted(list(original_category_set))
total_boxes = sum(len(v) for v in image_annotations.values())

print(f"   • Unique images found:            {len(image_annotations)}")
print(f"   • Total valid bounding boxes:     {total_boxes}")
print(f"   • Original category names found:  {ORIGINAL_CATEGORIES}")
print(f"   • Single-Class target category:   '{TARGET_CLASS}' (ID: 0)")
logger.info(f"Found {len(image_annotations)} images with {total_boxes} boxes. Original classes: {ORIGINAL_CATEGORIES}")


In [ ]:
# CELL 6 — Group Annotations & Apply Sample 1000 / Full Data Mode
print("=" * 70)
print(f"📊 [CELL 6] Applying mode selection: {MODE_TAG.upper()}")
print("=" * 70)

all_image_urls = sorted(list(image_annotations.keys()))
random.seed(RANDOM_SEED)

if SAMPLE_SIZE is not None and len(all_image_urls) > SAMPLE_SIZE:
    selected_image_urls = random.sample(all_image_urls, SAMPLE_SIZE)
    print(f"   🎯 Sample Mode Active: Selected {len(selected_image_urls)} images out of {len(all_image_urls)}.")
else:
    selected_image_urls = all_image_urls
    print(f"   🌐 Full Data Mode: Using all {len(selected_image_urls)} images.")

image_urls = sorted(selected_image_urls)
filtered_annotations = {u: image_annotations[u] for u in image_urls}
total_selected_boxes = sum(len(v) for v in filtered_annotations.values())

print(f"   • Total images in pipeline:        {len(image_urls)}")
print(f"   • Total bounding boxes in pipeline: {total_selected_boxes}")
logger.info(f"Mode [{MODE_TAG}]: {len(image_urls)} images, {total_selected_boxes} annotations.")


In [ ]:
# CELL 7 — Azure URL Helpers & Download Function
print("=" * 70)
print("☁️  [CELL 7] Defining Azure Blob download and caching functions...")
print("=" * 70)

RAW_IMAGES_DIR = PIPELINE_DIR / "raw_images"
RAW_IMAGES_DIR.mkdir(parents=True, exist_ok=True)

def extract_blob_info(url: str) -> Tuple[Optional[str], Optional[str], str]:
    parsed = urlparse(url)
    clean_path = parsed.path.lstrip("/")
    parts = clean_path.split("/", 1)
    filename = Path(clean_path).name.split("?")[0]
    if len(parts) == 2:
        return parts[0], parts[1], filename
    return None, None, filename

def download_image(url: str, output_dir: Path) -> Tuple[str, Optional[Path], Optional[str]]:
    container, blob_name, filename = extract_blob_info(url)
    dest_path = output_dir / filename
    if dest_path.exists() and dest_path.stat().st_size > 0:
        return url, dest_path, None
    if blob_service_client and container and blob_name:
        try:
            bc = blob_service_client.get_blob_client(container=container, blob=blob_name)
            with open(dest_path, "wb") as f:
                f.write(bc.download_blob().readall())
            return url, dest_path, None
        except Exception as e:
            pass
    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        with open(dest_path, "wb") as f:
            f.write(resp.content)
        return url, dest_path, None
    except Exception as e:
        return url, None, str(e)

print("✅ Image download functions ready.")


In [ ]:
# CELL 8 — Parallel Azure Image Downloads
print("=" * 70)
print(f"⬇️  [CELL 8] Downloading {len(image_urls)} images ({DOWNLOAD_WORKERS} threads)...")
print("=" * 70)

download_results: Dict[str, Dict[str, Any]] = {}
with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as executor:
    futures = {executor.submit(download_image, url, RAW_IMAGES_DIR): url for url in image_urls}
    done_count = 0
    for future in as_completed(futures):
        url, path, err = future.result()
        download_results[url] = {"local_path": path, "error": err}
        done_count += 1
        if done_count % 200 == 0 or done_count == len(image_urls):
            print(f"   • Progress: {done_count}/{len(image_urls)} processed...")

success_downloads = [u for u, res in download_results.items() if res["error"] is None]
failed_downloads = [u for u, res in download_results.items() if res["error"] is not None]

print(f"✅ Download Summary:")
print(f"   • Successfully ready: {len(success_downloads)} images")
print(f"   • Failed:             {len(failed_downloads)} images")
logger.info(f"Downloaded {len(success_downloads)} images, failed: {len(failed_downloads)}.")


In [ ]:
# CELL 9 — Train/Val/Test Split (80% / 10% / 10%)
print("=" * 70)
print("✂️  [CELL 9] Splitting dataset into Train (80%), Val (10%), Test (10%)...")
print("=" * 70)

valid_urls = [u for u in image_urls if download_results.get(u, {}).get("error") is None]
random.seed(RANDOM_SEED)
shuffled_urls = valid_urls.copy()
random.shuffle(shuffled_urls)

n_total = len(shuffled_urls)
n_train = int(n_total * TRAIN_RATIO)
n_val = int(n_total * VALID_RATIO)

train_urls = shuffled_urls[:n_train]
val_urls = shuffled_urls[n_train:n_train + n_val]
test_urls = shuffled_urls[n_train + n_val:]

split_summary_data = [
    {"Split": "Train", "Images": len(train_urls), "Boxes": sum(len(filtered_annotations[u]) for u in train_urls)},
    {"Split": "Val",   "Images": len(val_urls),   "Boxes": sum(len(filtered_annotations[u]) for u in val_urls)},
    {"Split": "Test",  "Images": len(test_urls),  "Boxes": sum(len(filtered_annotations[u]) for u in test_urls)},
    {"Split": "Total", "Images": n_total,         "Boxes": sum(len(filtered_annotations[u]) for u in valid_urls)}
]
split_df = pd.DataFrame(split_summary_data)

print(split_df.to_string(index=False))
logger.info(f"Split complete: Train={len(train_urls)}, Val={len(val_urls)}, Test={len(test_urls)}.")


In [ ]:
# CELL 10 — Create Split Folders & Organize Images
print("=" * 70)
print("📂 [CELL 10] Organizing split image directories...")
print("=" * 70)

for split_name in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    os.makedirs(split_name, exist_ok=True)

url_to_split_map = {}
for s_dir, u_list in [(TRAIN_DIR, train_urls), (VAL_DIR, val_urls), (TEST_DIR, test_urls)]:
    for url in u_list:
        url_to_split_map[url] = s_dir
        src_path = download_results[url]["local_path"]
        if src_path and src_path.exists():
            dest = os.path.join(s_dir, src_path.name)
            if not os.path.exists(dest):
                shutil.copy2(src_path, dest)

print(f"   • Train images directory: {TRAIN_DIR} ({len(os.listdir(TRAIN_DIR))} files)")
print(f"   • Val images directory:   {VAL_DIR} ({len(os.listdir(VAL_DIR))} files)")
print(f"   • Test images directory:  {TEST_DIR} ({len(os.listdir(TEST_DIR))} files)")
print("✅ Images organized successfully.")


In [ ]:
# CELL 11 — Build Single-Class COCO Annotations
print("=" * 70)
print(f"📝 [CELL 11] Building COCO format annotations for single class '{TARGET_CLASS}'...")
print("=" * 70)

categories_def = [{"id": 0, "name": TARGET_CLASS, "supercategory": "none"}]

def build_coco_for_split(urls: List[str], split_dir: str, ann_path: str) -> Dict[str, Any]:
    images_list = []
    annotations_list = []
    ann_id = 1

    for img_id_idx, url in enumerate(urls, start=1):
        local_path = download_results[url]["local_path"]
        if not (local_path and local_path.exists()):
            continue
        try:
            with Image.open(local_path) as im:
                width, height = im.size
        except Exception:
            width, height = 1920, 1080

        filename = local_path.name
        images_list.append({
            "id": img_id_idx,
            "file_name": filename,
            "width": int(width),
            "height": int(height),
            "original_url": url
        })

        for ann in filtered_annotations.get(url, []):
            x, y, w, h = ann["bbox"]
            annotations_list.append({
                "id": ann_id,
                "image_id": img_id_idx,
                "category_id": 0,
                "bbox": [round(x, 2), round(y, 2), round(w, 2), round(h, 2)],
                "area": round(w * h, 2),
                "iscrowd": 0,
                "segmentation": []
            })
            ann_id += 1

    coco_dict = {
        "info": {"description": f"RF-DETR Single-Class Dataset: {TARGET_CLASS}", "version": "1.0"},
        "licenses": [],
        "images": images_list,
        "annotations": annotations_list,
        "categories": categories_def
    }
    
    with open(ann_path, "w", encoding="utf-8") as f:
        json.dump(coco_dict, f, indent=2)
    print(f"   • Saved: {ann_path} ({len(images_list)} images, {len(annotations_list)} boxes)")
    return coco_dict

build_coco_for_split(train_urls, TRAIN_DIR, TRAIN_ANN)
build_coco_for_split(val_urls, VAL_DIR, VAL_ANN)
build_coco_for_split(test_urls, TEST_DIR, TEST_ANN)

print("✅ Single-Class COCO JSON files created.")


In [ ]:
# CELL 12 — Dataset Inspection (Summary Statistics Table)
def load_coco_info(annotation_path: str) -> dict:
    """Parse a COCO annotation file and return summary statistics."""
    with open(annotation_path) as f:
        data = json.load(f)
    categories = {c["id"]: c["name"] for c in data.get("categories", [])}
    num_images = len(data.get("images", []))
    num_anns = len(data.get("annotations", []))
    ann_per_cat: dict = {}
    for ann in data.get("annotations", []):
        cat_name = categories.get(ann["category_id"], "unknown")
        ann_per_cat[cat_name] = ann_per_cat.get(cat_name, 0) + 1
    return {
        "num_images": num_images,
        "num_anns": num_anns,
        "num_classes": len(categories),
        "categories": categories,
        "ann_per_cat": ann_per_cat,
        "raw": data,
    }

# Validate all annotation files exist
for split, path in [("train", TRAIN_ANN), ("val", VAL_ANN), ("test", TEST_ANN)]:
    assert os.path.exists(path), f"Missing annotation file for '{split}': {path}"
print("All annotation files found")

train_info = load_coco_info(TRAIN_ANN)
val_info = load_coco_info(VAL_ANN)
test_info = load_coco_info(TEST_ANN)

# NUM_CLASSES is derived from the training annotation file
NUM_CLASSES = train_info["num_classes"]

print()
print("=" * 58)
print(f"{'Split': <10} {'Images': >8} {'Annotations': >14} {'Classes': >9}")
print("-" * 58)
for name, info in [("train", train_info), ("val", val_info), ("test", test_info)]:
    print(f"{name: <10} {info['num_images']: >8} {info['num_anns']: >14} {info['num_classes']: >9}")
print("=" * 58)
print(f"\nClasses ({NUM_CLASSES} total): {train_info['categories']}")


In [ ]:
# CELL 13 — Visualize Samples Overlaid with Ground-Truth Bounding Boxes
def visualize_samples(annotation_path: str, image_dir: str, num_samples: int = 4, seed: int = 42):
    """Display random training samples overlaid with ground-truth bounding boxes."""
    rng = np.random.default_rng(seed)
    with open(annotation_path) as f:
        data = json.load(f)
    categories = {c["id"]: c["name"] for c in data["categories"]}
    img_id_map = {img["id"]: img for img in data["images"]}
    img_anns: dict = {}
    for ann in data["annotations"]:
        img_anns.setdefault(ann["image_id"], []).append(ann)
    
    valid_ids = [iid for iid in img_anns.keys() if os.path.exists(os.path.join(image_dir, img_id_map[iid]["file_name"]))]
    if not valid_ids:
        print("No valid local images found to visualize.")
        return
    
    ids = rng.choice(valid_ids, size=min(num_samples, len(valid_ids)), replace=False)
    n = len(ids)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
    if n == 1:
        axes = [axes]
    cmap = plt.cm.get_cmap("tab20", max(NUM_CLASSES, 1))

    for ax, img_id in zip(axes, ids):
        meta = img_id_map[img_id]
        img_path = os.path.join(image_dir, meta["file_name"])
        if not os.path.exists(img_path):
            ax.text(0.5, 0.5, "image \nnot found", ha="center", va="center", transform=ax.transAxes)
            ax.axis("off")
            continue
        img = Image.open(img_path).convert("RGB")
        ax.imshow(img)
        for ann in img_anns.get(img_id, []):
            x, y, w, h = ann["bbox"]
            cat_id = ann["category_id"]
            color = cmap(cat_id % max(NUM_CLASSES, 1))
            rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor=color, facecolor="none")
            ax.add_patch(rect)
            ax.text(
                x, max(y - 4, 0), categories.get(cat_id, str(cat_id)),
                fontsize=8, color=color,
                bbox=dict(boxstyle="round,pad=0.1", fc="white", alpha=0.65, ec="none")
            )
        ax.set_title(meta["file_name"][:20], fontsize=8)
        ax.axis("off")

    plt.suptitle("Training Set Samples - Ground Truth Boxes", fontsize=12, y=1.01)
    plt.tight_layout()
    save_path = os.path.join(LOG_DIR, "sample_images.png")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {save_path}")

visualize_samples(TRAIN_ANN, TRAIN_DIR)


In [ ]:
# CELL 14 — Model Initialization & Reinitialize Detection Head Patch
# 1. Detection Head Patch
def _fixed_reinitialize(self, num_classes):
    lwdetr = self.model  # the actual LWDETR nn.Module
    base = lwdetr.class_embed.weight.shape[0]
    if base == 0:
        in_features = lwdetr.class_embed.in_features
        device = next(lwdetr.parameters()).device
        lwdetr.class_embed = nn.Linear(in_features, num_classes).to(device)
        nn.init.normal_(lwdetr.class_embed.weight, std=0.01)
        nn.init.zeros_(lwdetr.class_embed.bias)
        print(f" class_embed reinitialized from scratch → ({in_features} → {num_classes})")
        return
    num_repeats = int(math.ceil(num_classes / base))
    lwdetr.class_embed.weight.data = lwdetr.class_embed.weight.data.repeat(num_repeats, 1)[:num_classes]
    if lwdetr.class_embed.bias is not None:
        lwdetr.class_embed.bias.data = lwdetr.class_embed.bias.data.repeat(num_repeats)[:num_classes]

rfdetr_main.Model.reinitialize_detection_head = _fixed_reinitialize
print("reinitialize_detection_head patched")

# 2. Model Instantiation
ModelClass = RFDETRLarge if MODEL_SIZE == "large" else RFDETRBase
model = ModelClass(
    num_classes=NUM_CLASSES,
    pretrain_weights=PRETRAINED_WEIGHTS,
    resolution=RESOLUTION
)
print(f"RF-DETR-{MODEL_SIZE.capitalize()} initialized with resolution={RESOLUTION}, num_classes={NUM_CLASSES}")

# Build ordered list of class names from COCO categories
class_names = [
    train_info["categories"][cid]
    for cid in sorted(train_info["categories"].keys())
]
print(f"class_names: {class_names}")


In [ ]:
# CELL 15 — PyTorch Dataset & DataLoader
class COCODetectionDataset(Dataset):
    """Loads COCO-format annotations and converts boxes to normalised [cx, cy, w, h]."""
    def __init__(self, img_dir: str, ann_file: str, resolution: int = 560):
        with open(ann_file) as f:
            data = json.load(f)
        self.img_dir = img_dir
        self.resolution = resolution
        cats = sorted(c["id"] for c in data["categories"])
        self.cat_to_id = {c: i for i, c in enumerate(cats)}
        ann_by_img: dict = {}
        for ann in data["annotations"]:
            ann_by_img.setdefault(ann["image_id"], []).append(ann)
        # Only keep images that have at least one annotation
        self.samples = [
            (img, ann_by_img[img["id"]])
            for img in data["images"] if img["id"] in ann_by_img
        ]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_meta, anns = self.samples[idx]
        img_path = os.path.join(self.img_dir, img_meta["file_name"])
        img = Image.open(img_path).convert("RGB")
        orig_w, orig_h = img.size
        img = img.resize((self.resolution, self.resolution), Image.BILINEAR)
        img_t = TF.to_tensor(img)
        img_t = TF.normalize(
            img_t,
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
        boxes, labels = [], []
        for ann in anns:
            x, y, w, h = ann["bbox"]
            if w <= 0 or h <= 0:
                continue
            cx = float(np.clip((x + w / 2) / orig_w, 0, 1))
            cy = float(np.clip((y + h / 2) / orig_h, 0, 1))
            bw = float(np.clip(w / orig_w, 0, 1))
            bh = float(np.clip(h / orig_h, 0, 1))
            boxes.append([cx, cy, bw, bh])
            labels.append(self.cat_to_id[ann["category_id"]])
        return img_t, {
            "boxes": torch.tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0, 4)),
            "labels": torch.tensor(labels, dtype=torch.long) if labels else torch.zeros(0, dtype=torch.long),
            "image_id": torch.tensor([img_meta["id"]]),
            "orig_size": torch.tensor([orig_h, orig_w]),
            "size": torch.tensor([self.resolution, self.resolution]),
        }

def collate_fn(batch):
    images = torch.stack([item[0] for item in batch])
    targets = [item[1] for item in batch]
    return images, targets

print("COCODetectionDataset & collate_fn ready.")


In [ ]:
# CELL 16 — Setup Criterion, DataLoaders, AdamW & BestValLossCheckpointSD
# 1. Build Criterion using RF-DETR defaults
args = copy.deepcopy(model.model.args)
args.num_classes = NUM_CLASSES
args.device = "cuda" if torch.cuda.is_available() else "cpu"
criterion, _ = build_criterion_and_postprocessors(args)

# 2. Device & actual LWDETR nn.Module
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lwdetr = model.model.model  # actual LWDETR nn.Module
lwdetr.to(device)
criterion.to(device)

# 3. Data Loaders
train_ds = COCODetectionDataset(TRAIN_DIR, TRAIN_ANN, RESOLUTION)
val_ds = COCODetectionDataset(VAL_DIR, VAL_ANN, RESOLUTION)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, collate_fn=collate_fn, drop_last=True, pin_memory=True
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, collate_fn=collate_fn, pin_memory=True
)

# 4. Optimizer & Cosine Scheduler
optimizer = torch.optim.AdamW(
    lwdetr.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
    eta_min=1e-6
)

# 5. BestValLossCheckpointSD Class
class BestValLossCheckpointSD:
    """State-dict based checkpoint - no rfdetr API dependency."""
    def __init__(self, checkpoint_dir: str, module: nn.Module, verbose: bool = True):
        self.ckpt_dir = Path(checkpoint_dir)
        self.ckpt_dir.mkdir(parents=True, exist_ok=True)
        self.module = module
        self.verbose = verbose
        self.best_loss = float("inf")
        self.best_epoch = -1
        self.history = []
        self.log_path = self.ckpt_dir / "training_log.json"
        self.best_path = self.ckpt_dir / "best_model.pth"

    def step(self, epoch: int, train_loss: float, val_loss: float):
        self.history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
        })
        improved = val_loss < self.best_loss
        if improved:
            self.best_loss = val_loss
            self.best_epoch = epoch
            torch.save(self.module.state_dict(), str(self.best_path))
            versioned = self.ckpt_dir / f"epoch_{epoch:04d}_val{val_loss:.4f}.pth"
            torch.save(self.module.state_dict(), str(versioned))
        self._write_log()
        if self.verbose:
            marker = " <-- best" if improved else ""
            print(f" epoch {epoch:>4d} | train {train_loss:.4f} | val {val_loss:.4f} | best {self.best_loss:.4f}{marker}")

    def _write_log(self):
        with open(self.log_path, "w") as f:
            json.dump({
                "best_epoch": self.best_epoch,
                "best_val_loss": self.best_loss,
                "history": self.history
            }, f, indent=2)

    def load_best(self):
        self.module.load_state_dict(torch.load(str(self.best_path), map_location="cpu"))
        print(f"Loaded best checkpoint - epoch {self.best_epoch}, val_loss {self.best_loss:.6f}")

ckpt = BestValLossCheckpointSD(CHECKPOINT_DIR, lwdetr, verbose=True)
print_vram_usage("Pre-Training Setup")


In [ ]:
# CELL 17 — Training Step (AMP, Grad Accumulation) & compute_map Evaluation
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

def run_epoch(model, loader, criterion, optimizer, device, is_train: bool, epoch: int, num_epochs: int) -> float:
    model.train(is_train)
    phase = "train" if is_train else "val"
    total, count = 0.0, 0
    pbar = tqdm(loader, desc=f"Epoch {epoch:>4}/{num_epochs} [{phase}]", leave=False, unit="batch", dynamic_ncols=True)

    with torch.set_grad_enabled(is_train):
        for step, (images, targets) in enumerate(pbar):
            images = images.to(device, non_blocking=True)
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            # AMP forward pass
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                outputs = model(images)
                loss_dict = criterion(outputs, targets)
                loss = sum(loss_dict[k] * criterion.weight_dict[k] for k in loss_dict if k in criterion.weight_dict)

            if is_train:
                if step % GRAD_ACCUMULATION == 0:
                    optimizer.zero_grad()
                scaler.scale(loss).backward()
                if (step + 1) % GRAD_ACCUMULATION == 0 or (step + 1) == len(loader):
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.1)
                    scaler.step(optimizer)
                    scaler.update()

            total += loss.item()
            count += 1
            pbar.set_postfix({"loss": f"{total / count:.4f}"})

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    pbar.close()
    return total / max(count, 1)

def box_cxcywh_to_xyxy(boxes: torch.Tensor) -> torch.Tensor:
    """Convert normalised [cx, cy, w, h] → [x1, y1, x2, y2]."""
    cx, cy, w, h = boxes.unbind(-1)
    return torch.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], dim=-1)

def compute_map(model, loader, device, img_size: int, threshold: float = 0.3) -> dict:
    """Run inference on loader and return COCO-style mAP metrics."""
    model.eval()
    metric = MeanAveragePrecision(iou_type="bbox", class_metrics=False)
    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                outputs = model(images)
            pred_logits = outputs["pred_logits"]  # [B, Q, C]
            pred_boxes = outputs["pred_boxes"]    # [B, Q, 4] normalised cxcywh
            preds_list, tgts_list = [], []
            for i in range(len(images)):
                scores_all = pred_logits[i].sigmoid()
                scores, lbs = scores_all.max(-1)
                keep = scores > threshold
                boxes_pred = box_cxcywh_to_xyxy(pred_boxes[i][keep]) * img_size
                preds_list.append({
                    "boxes": boxes_pred.cpu().float(),
                    "scores": scores[keep].cpu().float(),
                    "labels": lbs[keep].cpu(),
                })
                gt_boxes = box_cxcywh_to_xyxy(targets[i]["boxes"]) * img_size
                tgts_list.append({
                    "boxes": gt_boxes.cpu().float(),
                    "labels": targets[i]["labels"].cpu(),
                })
            metric.update(preds_list, tgts_list)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return metric.compute()

print("Training step and compute_map functions compiled.")


In [ ]:
# CELL 18 — Fine-Tuning Execution Loop
print("=" * 70)
print(f"🔥 [CELL 18] Starting Training for {EPOCHS} Epochs (Batch: {BATCH_SIZE}, Effective: {BATCH_SIZE*GRAD_ACCUMULATION})...")
print("=" * 70)

for epoch in range(1, EPOCHS + 1):
    tr_loss = run_epoch(lwdetr, train_loader, criterion, optimizer, device, is_train=True, epoch=epoch, num_epochs=EPOCHS)
    val_loss = run_epoch(lwdetr, val_loader, criterion, optimizer, device, is_train=False, epoch=epoch, num_epochs=EPOCHS)
    scheduler.step()
    ckpt.step(epoch, tr_loss, val_loss)

print("=" * 70)
print("✅ Training finished successfully!")
print_vram_usage("Post-Training Peak")


In [ ]:
# CELL 19 — Load Best Checkpoint & Compute Validation mAP
print("=" * 70)
print("🏆 [CELL 19] Loading best checkpoint and evaluating validation mAP...")
print("=" * 70)

ckpt.load_best()
val_map_results = compute_map(lwdetr, val_loader, device, img_size=RESOLUTION, threshold=0.3)

print("=" * 50)
print("Validation mAP Metrics:")
for k, v in val_map_results.items():
    if isinstance(v, torch.Tensor):
        print(f"   • {k:20s}: {v.item():.4f}")
    else:
        print(f"   • {k:20s}: {v}")
print("=" * 50)


In [ ]:
# CELL 20 — Export Best Model to model/ Directory
print("=" * 70)
print("💾 [CELL 20] Exporting best model checkpoint...")
print("=" * 70)

exported_path = os.path.join(FINAL_MODEL_DIR, f"best_model_{MODE_TAG}.pth")
if os.path.exists(ckpt.best_path):
    shutil.copy2(ckpt.best_path, exported_path)
    print(f"   • Canonical best model saved to: {exported_path} ({os.path.getsize(exported_path) / 1e6:.1f} MB)")
else:
    print(f"   ⚠️  Checkpoint not found at {ckpt.best_path}")

print("✅ Export completed.")


In [ ]:
# CELL 21 — Test Set Inference & Visual Predictions
print("=" * 70)
print(f"🎯 [CELL 21] Running inference on test split (Confidence={CONFIDENCE}, NMS={NMS_THRESHOLD})...")
print("=" * 70)

test_ds = COCODetectionDataset(TEST_DIR, TEST_ANN, RESOLUTION)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=1, collate_fn=collate_fn)

test_map_results = compute_map(lwdetr, test_loader, device, img_size=RESOLUTION, threshold=CONFIDENCE)
print("Test Set mAP Results:")
for k, v in test_map_results.items():
    if isinstance(v, torch.Tensor):
        print(f"   • {k:20s}: {v.item():.4f}")

# Visual sample test predictions using supervision
box_annotator = sv.BoxAnnotator(color=sv.Color.GREEN, thickness=2)
label_annotator = sv.LabelAnnotator(color=sv.Color.GREEN, text_scale=0.5, text_thickness=1)

lwdetr.eval()
preview_count = 0

with torch.no_grad():
    for images, targets in test_loader:
        if preview_count >= 3:
            break
        images = images.to(device)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            outputs = lwdetr(images)
        pred_logits = outputs["pred_logits"][0]
        pred_boxes = outputs["pred_boxes"][0]

        scores, _ = pred_logits.sigmoid().max(-1)
        keep = scores > CONFIDENCE
        if keep.sum() == 0:
            continue

        boxes_xyxy = (box_cxcywh_to_xyxy(pred_boxes[keep]) * RESOLUTION).cpu().numpy()
        scores_np = scores[keep].cpu().numpy()

        img_np = (images[0].permute(1, 2, 0).cpu().numpy() * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]))
        img_np = np.clip(img_np * 255, 0, 255).astype(np.uint8)

        detections = sv.Detections(xyxy=boxes_xyxy, confidence=scores_np)
        labels = [f"{TARGET_CLASS} {c:.2f}" for c in scores_np]
        annotated = box_annotator.annotate(scene=img_np.copy(), detections=detections)
        annotated = label_annotator.annotate(scene=annotated, detections=detections, labels=labels)

        plt.figure(figsize=(10, 6))
        plt.imshow(annotated)
        plt.title(f"Test Detection Preview ({len(boxes_xyxy)} '{TARGET_CLASS}' detected)")
        plt.axis("off")
        plt.show()
        preview_count += 1

print(f"✅ Previewed {preview_count} test detection results.")


In [ ]:
# CELL 22 — Save Test Predictions JSON
print("=" * 70)
print("💾 [CELL 22] Saving test predictions...")
print("=" * 70)

pred_out_file = os.path.join(INFERENCE_OUTPUT_DIR, f"test_predictions_{MODE_TAG}.json")
lwdetr.eval()
all_test_predictions = {}

with torch.no_grad():
    for images, targets in test_loader:
        img_id = targets[0]["image_id"].item()
        orig_h, orig_w = targets[0]["orig_size"].tolist()
        images = images.to(device)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            outputs = lwdetr(images)
        pred_logits = outputs["pred_logits"][0]
        pred_boxes = outputs["pred_boxes"][0]

        scores, _ = pred_logits.sigmoid().max(-1)
        keep = scores > CONFIDENCE

        boxes_xyxy = (box_cxcywh_to_xyxy(pred_boxes[keep]) * np.array([orig_w, orig_h, orig_w, orig_h])).cpu().tolist()
        scores_list = scores[keep].cpu().tolist()

        all_test_predictions[str(img_id)] = {
            "boxes_xyxy": boxes_xyxy,
            "confidence": scores_list,
            "orig_size": [orig_h, orig_w]
        }

with open(pred_out_file, "w", encoding="utf-8") as f:
    json.dump({
        "pipeline": PIPELINE_NAME,
        "mode": MODE_TAG,
        "target_class": TARGET_CLASS,
        "confidence_threshold": CONFIDENCE,
        "predictions": all_test_predictions
    }, f, indent=2)

print(f"✅ Saved test predictions to: {pred_out_file}")
logger.info(f"Saved test predictions to {pred_out_file}")


In [ ]:
# CELL 23 — Final Execution Summary
print("=" * 70)
print(f"🏁 [CELL 23] Execution Summary: {PIPELINE_NAME}")
print("=" * 70)
print(f"   • Pipeline Mode:        {MODE_TAG.upper()}")
print(f"   • Target Class:         '{TARGET_CLASS}' (Single Class ID: 0)")
print(f"   • Architecture:         RF-DETR-{MODEL_SIZE.capitalize()} (resolution={RESOLUTION})")
print(f"   • Effective Batch Size: {BATCH_SIZE * GRAD_ACCUMULATION} (Batch={BATCH_SIZE}, Accum={GRAD_ACCUMULATION})")
print(f"   • Learning Rate:        {LEARNING_RATE} (Weight Decay: {WEIGHT_DECAY})")
print(f"   • Dataset Folder:       {DATASET_DIR}")
print(f"   • Checkpoint Folder:    {CHECKPOINT_DIR}")
print(f"   • Exported Model:       {exported_path}")
print(f"   • Test Predictions:     {pred_out_file}")
print("=" * 70)
print("🎉 Single-Class Pipeline execution completed successfully!")
logger.info("Single-class pipeline completed successfully.")
